In [8]:
import numpy as np
from circular_mode import CircularMode

In [9]:
freq = 20e9      # 20 GHz
a = 0.010        # rayon guide : 10 mm
TE11 = CircularMode("TE", 1, 1, a, freq)
TE11.export_to_vtk(filename="TE11.vts")

Export VTS réussi : 'TE11.vts' (Grille 3D 21x51x100).


In [10]:
def rtz_to_xyz(r, theta, z):
    return r * np.cos(theta), r * np.sin(theta), z

def xyz_to_rtz(x, y, z):
    return np.sqrt(x ** 2 + y ** 2), np.atan2(y, x), z

def spatial_gradient(mode: CircularMode, r: float, theta: float, z: float, h=1e-6):
    E ,H = mode.field_xyz(r, theta, z)
    x, y, z = rtz_to_xyz(r, theta, z)
    # x derivative
    x_h = x + h
    r_dx, t_dx, z_dx = xyz_to_rtz(x_h, y, z)
    E_dx, H_dx = mode.field_xyz(r_dx, t_dx, z_dx)
    E_dx = (E_dx - E) / h
    H_dx = (H_dx - H) / h
    # y derivative
    y_h = y + h
    r_dy, t_dy, z_dy = xyz_to_rtz(x, y_h, z)
    E_dy, H_dy = mode.field_xyz(r_dy, t_dy, z_dy)
    E_dy = (E_dy - E) / h
    H_dy = (H_dy - H) / h
    # z derivative
    z_h = z + h
    r_dz, t_dz, z_dz = xyz_to_rtz(x, y, z_h)
    E_dz, H_dz = mode.field_xyz(r_dz, t_dz, z_dz)
    E_dz = (E_dz - E) / h
    H_dz = (H_dz - H) / h
    return E_dx, E_dy, E_dz, H_dx, H_dy, H_dz

def rot(f_dx, f_dy, f_dz):
    return np.array([
        f_dy[2] - f_dz[1],
        f_dz[0] - f_dx[2],
        f_dx[1] - f_dy[0]
    ])

In [11]:
# check Maxwell equation are satisfied
# - select random point in the guide:
r = np.random.rand() * TE11.R
theta = 2 * np.pi * np.random.rand()
z = 5. * TE11.R * np.random.rand()
E ,H = TE11.field_xyz(r, theta, z)
E_dx, E_dy, E_dz, H_dx, H_dy, H_dz = spatial_gradient(TE11, r, theta, z, h=1e-8)
rot_E = rot(E_dx, E_dy, E_dz)
rot_H = rot(H_dx, H_dy, H_dz)

In [12]:
from circular_mode import mu0, eps0
res_E = rot_E + 1j * TE11.omega * mu0 * H
print(rot_E)
print(1j * TE11.omega * mu0 * H)
print(res_E/np.linalg.norm(rot_E))

[ 25440.15644617-147417.39101964j   -842.12165632  +4879.81971453j
 -30918.65583826  -5335.76312129j]
[-25440.43398273+147417.34302448j    842.13083993  -4879.81812673j
  30918.704325    +5335.77149116j]
[-1.81477695e-06-3.13834417e-07j  6.00504600e-08+1.03824121e-08j
  3.17048748e-07+5.47294730e-08j]


In [13]:
res_H = rot_H - 1j * TE11.omega * eps0 * E
print(res_H)
print(res_H/np.linalg.norm(rot_H))

[-2.33264185e-05+4.47037262e-06j -6.57535540e-04-1.39375096e-04j
  4.25715019e-05+7.34551309e-06j]
[-5.27440246e-08+1.01080859e-08j -1.48677220e-06-3.15144970e-07j
  9.62596268e-08+1.66091474e-08j]


In [14]:
# check boundary conditions Ez(r=a)=0, Etheta(r=a)=0, Hr(r=a)=0
r = TE11.R
theta = 2 * np.pi * np.random.rand()
z = 5. * TE11.R * np.random.rand()
E, H = TE11.field_rtz(r, theta, z)
print(E)
print(H)

[5.74086241e+01+1.63261781e+02j 3.64657660e-14+1.03703337e-13j
 0.00000000e+00+0.00000000e+00j]
[-8.69577793e-17-2.47295282e-16j  1.36898988e-01+3.89320819e-01j
 -4.22465314e-01+1.48553766e-01j]
